# Text-to-SQL Analyst Agent: try it in Colab

Ask questions in plain English about a database and get back SQL and a result table. A fine-tuned `Qwen2.5-Coder-3B` model writes the query, and a LangGraph agent validates it, runs it read-only, and retries if it fails.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhruvish-28/Analyst-Agent/blob/main/notebooks/analyst_agent_colab.ipynb)

**How to use**
1. Runtime → Change runtime type → **T4 GPU** (free tier is fine).
2. Runtime → **Run all**. The first run takes about 5 minutes, mostly downloading the ~6 GB base model.
3. When the last cell finishes, a query interface appears below it. Pick a sample database or upload your own file.

**Your own data:** upload a `.sqlite` / `.db` file, a `.csv` / `.tsv` file, or an `.xlsx` workbook (each sheet becomes a table). Max 50 MB. Files stay inside this Colab session and are opened read-only.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In the menu choose Runtime -> Change runtime type -> T4 GPU, "
        "then use Runtime -> Run all again."
    )

props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | {props.total_memory / 1024**3:.1f} GB | compute capability {props.major}.{props.minor}")

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/Dhruvish-28/Analyst-Agent.git"
REPO_DIR = "/content/Analyst-Agent"

# Safe to re-run: only clones if the folder is missing.
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

!pip install -q "transformers>=4.45" "peft>=0.13" "accelerate>=0.34" "bitsandbytes>=0.43.3" sqlglot langgraph rich gradio pandas openpyxl

adapter = os.path.join(REPO_DIR, "output", "final_adapter", "adapter_model.safetensors")
assert os.path.getsize(adapter) > 10_000_000, "Adapter file looks incomplete (is it a Git LFS pointer?)."
print("Repo and dependencies ready.")

In [ ]:
import sys, torch

SRC = os.path.join(REPO_DIR, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO_DIR)

# The repo has a module called `evaluate`; make sure it is not shadowed by Hugging Face's package of the same name.
sys.modules.pop("evaluate", None)

# T4 GPUs (compute capability 7.5) have no native bfloat16. The model was trained with bf16,
# so on older GPUs we switch this notebook's copy of the code to float16 for inference.
if torch.cuda.get_device_capability()[0] < 8:
    path = os.path.join(SRC, "evaluate.py")
    text = open(path).read()
    open(path, "w").write(text.replace("torch.bfloat16", "torch.float16"))
    print("Older GPU detected: using float16 instead of bfloat16.")
else:
    print("GPU supports bfloat16: no changes needed.")

In [ ]:
from pathlib import Path
from setup_eval_data import ensure_database

# A few of the Spider benchmark databases, so you can try the agent right away.
SAMPLE_IDS = ["singer", "concert_singer", "world_1", "car_1", "flight_2", "dog_kennels", "tvshow"]
SPIDER_DIR = Path(REPO_DIR) / "data" / "spider_db"

SAMPLE_DBS = {}
for db_id in SAMPLE_IDS:
    try:
        SAMPLE_DBS[db_id] = str(ensure_database(db_id, SPIDER_DIR))
    except Exception as e:
        print(f"Skipped {db_id}: {e}")

print(f"{len(SAMPLE_DBS)} sample databases ready: {', '.join(SAMPLE_DBS)}")

In [ ]:
from agent import get_generator

# Downloads the base model (~6 GB, first run only), loads it in 4-bit, and attaches the fine-tuned adapter.
generator = get_generator()
assert "evaluate.py" in sys.modules["evaluate"].__file__ and SRC in sys.modules["evaluate"].__file__

print(f"Model ready. VRAM in use: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

### Helpers: loading your own data and fitting the schema in the prompt

The model was trained on prompts of at most 1,024 tokens, so a big schema has to be trimmed. If your database has too many tables, only the ones that match your question (plus tables they reference through foreign keys) are sent to the model.

In [ ]:
import re, shutil, sqlite3, time
import pandas as pd
import sql_tools
from agent import build_agent_graph

UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
MAX_UPLOAD_MB = 50
SCHEMA_TOKEN_BUDGET = 700   # leaves room for the system prompt, the question and the answer inside 1,024 tokens


def _clean(name, fallback="col"):
    s = re.sub(r"[^0-9a-zA-Z]+", "_", str(name).strip()).strip("_").lower()
    if not s:
        s = fallback
    if s[0].isdigit():
        s = f"{fallback}_{s}"
    return s


def _unique(names):
    used, out = set(), []
    for n in names:
        cand, i = n, 2
        while cand in used:
            cand = f"{n}_{i}"
            i += 1
        used.add(cand)
        out.append(cand)
    return out


def _read_csv(path, sep):
    try:
        return pd.read_csv(path, sep=sep)
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=sep, encoding="latin-1")


def load_database(path):
    # Turns an uploaded .sqlite/.db, .csv/.tsv or .xlsx file into a SQLite file the agent can query.
    p = Path(path)
    size_mb = p.stat().st_size / 1e6
    if size_mb > MAX_UPLOAD_MB:
        raise ValueError(f"The file is {size_mb:.0f} MB; the limit is {MAX_UPLOAD_MB} MB.")

    ext = p.suffix.lower()
    stem = _clean(p.stem, "data")
    out = UPLOAD_DIR / f"{stem}.sqlite"
    if out.exists():
        out.unlink()

    if ext in (".sqlite", ".sqlite3", ".db"):
        with open(p, "rb") as f:
            if f.read(16) != b"SQLite format 3\x00":
                raise ValueError("This file is not a valid SQLite database.")
        shutil.copy2(p, out)
    elif ext in (".csv", ".tsv", ".xlsx", ".xlsm"):
        if ext in (".csv", ".tsv"):
            frames = {stem: _read_csv(p, "\t" if ext == ".tsv" else ",")}
        else:
            frames = pd.read_excel(p, sheet_name=None)
        con = sqlite3.connect(out)
        try:
            tables = _unique([_clean(n, "sheet") for n in frames])
            for table, df in zip(tables, frames.values()):
                df = df.dropna(how="all").dropna(axis=1, how="all")
                if df.empty:
                    continue
                df.columns = _unique([_clean(c) for c in df.columns])
                df.to_sql(table, con, index=False, if_exists="replace")
            con.commit()
        finally:
            con.close()
    else:
        raise ValueError(f"Unsupported file type '{ext}'. Use .sqlite, .db, .csv, .tsv or .xlsx.")

    if not sql_tools.get_schema(out).strip():
        raise ValueError("No tables were found in this file.")
    return str(out)


def _split_words(text):
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(text))
    words = {w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2}

    def stem(w):
        if w.endswith("ies") and len(w) > 4:
            return w[:-3] + "y"
        if w.endswith("s") and not w.endswith("ss") and len(w) > 3:
            return w[:-1]
        return w

    return {stem(w) for w in words}


def build_schema(db_path, question, tokenizer, budget=SCHEMA_TOKEN_BUDGET):
    # Returns (schema_text, dropped_table_names). Sends everything if it fits, otherwise the most relevant tables.
    con = sql_tools.open_readonly(db_path)
    try:
        rows = con.execute(
            "SELECT name, sql FROM sqlite_master WHERE type='table' "
            "AND name NOT LIKE 'sqlite_%' AND sql IS NOT NULL ORDER BY name"
        ).fetchall()
        cols, fks = {}, {}
        for name, _ in rows:
            safe = name.replace('"', '""')
            cols[name] = [r[1] for r in con.execute(f'PRAGMA table_info("{safe}")')]
            try:
                fks[name] = {r[2] for r in con.execute(f'PRAGMA foreign_key_list("{safe}")')}
            except sqlite3.Error:
                fks[name] = set()
    finally:
        con.close()

    ddl = {n: s.strip().rstrip(";") + ";" for n, s in rows}
    names = [n for n, _ in rows]

    def ntok(text):
        return len(tokenizer(text)["input_ids"])

    if ntok("\n\n".join(ddl[n] for n in names)) <= budget:
        return "\n\n".join(ddl[n] for n in names), []

    q = _split_words(question)

    def score(n):
        return 3 * len(q & _split_words(n)) + sum(len(q & _split_words(c)) for c in cols[n])

    ranked = sorted(names, key=score, reverse=True)
    if score(ranked[0]) == 0:
        ranked = names   # nothing in the question matches any name: just fill the budget in order

    chosen, used = [], 0

    def try_add(n):
        nonlocal used
        if n in chosen:
            return
        cost = ntok(ddl[n]) + 2
        if not chosen or used + cost <= budget:
            chosen.append(n)
            used += cost

    for n in ranked:
        if chosen and score(n) == 0 and ranked is not names:
            break
        try_add(n)
        for m in sorted(fks[n]):
            if m in ddl:
                try_add(m)

    kept = [n for n in names if n in chosen]
    dropped = [n for n in names if n not in chosen]
    return "\n\n".join(ddl[n] for n in kept), dropped


_graph = None


def ask(question, db_path, strategy="resample", max_retries=3):
    global _graph
    gen = get_generator()
    if _graph is None:
        _graph = build_agent_graph()
    schema, dropped = build_schema(db_path, question, gen.tokenizer)
    state = {
        "question": question,
        "db_path": db_path,
        "schema_ddl": schema,
        "sql": "",
        "columns": [],
        "rows": [],
        "is_truncated": False,
        "error_message": None,
        "attempts": [],
        "retry_count": 0,
        "max_retries": int(max_retries),
        "strategy": strategy,
        "status": "pending",
    }
    start = time.perf_counter()
    result = _graph.invoke(state)
    return result, schema, dropped, time.perf_counter() - start

### Query interface

Choose a source, type a question, and press **Run**. Tip: complete questions ("What is the population of each country?") work better than fragments ("population of each country").

In [ ]:
import gradio as gr

UPLOAD_TYPES = [".sqlite", ".sqlite3", ".db", ".csv", ".tsv", ".xlsx", ".xlsm"]


def run_ui(source, sample_name, upload, question, strategy, retries):
    empty_df = pd.DataFrame()
    try:
        if not question or not question.strip():
            return "Type a question first.", "", empty_df, "", ""
        if source == "My own file":
            if upload is None:
                return "Upload a file first, or switch to a sample database.", "", empty_df, "", ""
            db_path = load_database(upload)
        else:
            if sample_name not in SAMPLE_DBS:
                return "Pick a sample database.", "", empty_df, "", ""
            db_path = SAMPLE_DBS[sample_name]

        result, schema, dropped, latency = ask(question.strip(), db_path, strategy, retries)
    except Exception as e:
        return f"**Error:** {e}", "", empty_df, "", ""

    ok = result.get("status") == "ok"
    attempts = result.get("attempts", [])
    rows = [list(r) for r in result.get("rows", [])]
    df = pd.DataFrame(rows, columns=result.get("columns") or None) if rows else empty_df

    status = f"**{'Success' if ok else 'Failed'}** | {latency:.1f} s | {len(attempts)} attempt(s) | {len(rows)} row(s)"
    if result.get("is_truncated"):
        status += " (capped at 200)"
    if ok and len(attempts) > 1:
        status += " | recovered after a retry"
    if not ok and result.get("error_message"):
        status += f"\n\nLast error: `{result['error_message']}`"
    if dropped:
        status += f"\n\nTables left out to fit the model's context: {', '.join(dropped)}"

    trace = "\n\n".join(
        f"**Attempt {a['attempt']}**: {'failed' if a.get('error') else 'ran cleanly'}\n```sql\n{a['sql']}\n```"
        + (f"\nError: `{a['error']}`" if a.get("error") else "")
        for a in attempts
    )
    return status, result.get("sql", ""), df, trace, schema


with gr.Blocks(title="Text-to-SQL Analyst Agent") as demo:
    gr.Markdown("## Text-to-SQL Analyst Agent")
    with gr.Row():
        with gr.Column(scale=2):
            source = gr.Radio(["Sample database", "My own file"], value="Sample database", label="Data source")
            sample = gr.Dropdown(list(SAMPLE_DBS), value=next(iter(SAMPLE_DBS), None), label="Sample database")
            upload = gr.File(label="Your file (.sqlite, .db, .csv, .tsv, .xlsx; max 50 MB)",
                             file_types=UPLOAD_TYPES, type="filepath")
            strategy = gr.Radio(["resample", "error_feedback"], value="resample", label="Retry strategy")
            retries = gr.Slider(1, 5, value=3, step=1, label="Max attempts")
        with gr.Column(scale=3):
            question = gr.Textbox(label="Your question", lines=3,
                                  placeholder="How many singers are there in each country?")
            run = gr.Button("Run", variant="primary")
            status = gr.Markdown()
            sql_out = gr.Code(language="sql", label="Generated SQL", interactive=False)
            table = gr.Dataframe(label="Result", interactive=False, wrap=True)
            with gr.Accordion("Attempt trace", open=False):
                trace = gr.Markdown()
            with gr.Accordion("Schema sent to the model", open=False):
                schema_box = gr.Code(language="sql", interactive=False)

    inputs = [source, sample, upload, question, strategy, retries]
    outputs = [status, sql_out, table, trace, schema_box]
    run.click(run_ui, inputs, outputs)
    question.submit(run_ui, inputs, outputs)

demo.launch()

### Good to know

- **Accuracy on your own data will usually be lower than on the Spider benchmark.** Real-world schemas are messier, and a 3B model struggles most with multi-table joins and ambiguous column names. Always read the generated SQL before trusting a number.
- **SQLite SQL only.** Uploaded CSV and Excel files are converted to SQLite, with column names cleaned to `snake_case`. Connecting to Postgres or other servers is not supported in this notebook.
- **Everything stays in this session.** Nothing is sent to an outside API; uploaded files are deleted when the Colab runtime resets.
- If the app stops responding after a long idle period, Colab may have reset the runtime. Use Runtime → Run all again.